# **Full Inference + Solution Mapping Code**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 🔹 Load base BanglaBERT from Hugging Face (not local path)
model_name = "sagorsarker/bangla-bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=33)
# change num_labels = number of classes in your task

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("✅ Base BanglaBERT loaded successfully on", device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/491 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/660M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sagorsarker/bangla-bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Base BanglaBERT loaded successfully on cpu


# Load a pretrained BanglaBERT

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load base BanglaBERT (pretrained only, no fine-tuning yet)
model_name = "sagorsarker/bangla-bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=33)
# ⚠️ put the number of causes you want to predict (len(cause2solution_bangla))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("✅ BanglaBERT loaded on", device)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at sagorsarker/bangla-bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ BanglaBERT loaded on cpu


# **cause → solution mapping**

In [ ]:
cause2solution_bangla = {
      "vehicle_emissions": "সার্বজনীন পরিবহন বৃদ্ধি করুন এবং ব্যক্তিগত গাড়ির ব্যবহার সীমিত করুন।",
    "industrial_emissions": "কারখানায় বাতাসের ফিল্টার ব্যবহার এবং নির্গমন নিয়ন্ত্রণ নীতি প্রয়োগ করুন।",
    "brick_kilns": "পরিবেশ বান্ধব ইটভাটা প্রযুক্তি ব্যবহার এবং বসতবাড়ি থেকে দূরে স্থানান্তর করুন।",
    "open_waste_burning": "খোলা বর্জ্য জ্বালানো বন্ধ করুন এবং বর্জ্য সংগ্রহের ব্যবস্থা করুন।",
    "construction_and_road_dust": "নির্মাণ স্থলে পানি ছিটিয়ে ধুলো কমান এবং রাস্তা ঢেকে রাখুন।",
    "plastic_waste_in_water": "প্লাস্টিক ব্যবস্থাপনা উন্নত করুন এবং পুনঃব্যবহারযোগ্য পণ্য ব্যবহার করুন।",
    "solid_waste_dumping": "সংগঠিত বর্জ্য নিষ্পত্তি ও পুনঃপ্রক্রিয়াকরণ কেন্দ্র তৈরি করুন।",
    "industrial_wastewater_discharge": "শিল্প কারখানার বর্জ্য জলে ফেলা নিয়ন্ত্রণ করুন।",
    "sewage_and_drainage": "পানি নিষ্কাশন ও নিকাশী ব্যবস্থার উন্নতি করুন।",
    "arsenic_contamination": "পানি পরীক্ষার ব্যবস্থা বৃদ্ধি করুন এবং আর্সেনিক দূষণ রোধ করুন।",
    "fossil_fuel_burning": "জ্বালানি দাহ কমান এবং নবায়নযোগ্য শক্তি ব্যবহার করুন।",
    "deforestation_land_use_change": "বন সংরক্ষণ করুন এবং অবৈধ চাষ ও কেটে ফেলা রোধ করুন।",
    "coal_power_plants": "কয়লা ব্যবহার কমান এবং পরিচ্ছন্ন শক্তি উৎস ব্যবহার করুন।",
    "traffic_noise": "যানবাহনের গতি নিয়ন্ত্রণ করুন এবং শহরে যানবাহন সংখ্যা কমান।",
    "industrial_noise": "শিল্প প্রতিষ্ঠানগুলিতে শব্দ নিয়ন্ত্রণ ব্যবস্থা ব্যবহার করুন।",
    "construction_noise": "নির্মাণ সময় নির্দিষ্ট ঘণ্টায় কাজ করুন এবং শব্দ কমানোর যন্ত্র ব্যবহার করুন।",
    "loudspeakers_and_social_noise": "সামাজিক ও জনসাধারণের শব্দ নিয়ন্ত্রণ করুন।",
    "heavy_metals_contamination": "শিল্প বর্জ্য নিয়ন্ত্রণ এবং ধাতু দূষণ রোধ করুন।",
    "agricultural_burning": "কৃষি জ্বালানি কমান এবং পরিবেশবান্ধব কৃষি পদ্ধতি অনুসরণ করুন।",
    "urban_street_lighting": "শহরের আলো নিয়ন্ত্রণ এবং অনাবশ্যক লাইট বন্ধ করুন।",
    "industrial_lighting": "শিল্প প্রতিষ্ঠানে আলো কমানো এবং দক্ষ আলো ব্যবস্থাপনা করুন।",
    "advertisement_billboards": "বিজ্ঞাপন বোর্ড কমান এবং পরিবেশ বান্ধব উপায় ব্যবহার করুন।",
    "glacier_melting": "গ্লেসিয়ারের সংরক্ষণ এবং জলবায়ু পরিবর্তন নিয়ন্ত্রণ।",
    "illegal_logging": "অবৈধ চুরি বন্ধ করুন এবং বন সংরক্ষণ বৃদ্ধি করুন।",
    "land_use_change": "ভূমি ব্যবহারের পরিকল্পনা করুন এবং বনাঞ্চল সংরক্ষণ করুন।",
    "agrochemical_overuse": "কৃষিতে রাসায়নিক ব্যবহার সীমিত করুন এবং জৈব চাষ প্রচার করুন।",
    "industrial_waste_dumping": "শিল্প বর্জ্য যথাযথভাবে নিষ্পত্তি করুন।",
    "landfill_leachate": "ল্যান্ডফিল থেকে দূষণ নিয়ন্ত্রণ করুন।",
    "habitat_loss": "প্রাণীর আবাসস্থল সংরক্ষণ করুন।",
    "illegal_poaching": "অবৈধ শিকার রোধ করুন।",
    "climate_change": "জলবায়ু পরিবর্তনের প্রভাব কমাতে ব্যবস্থা নিন।",
    "global_warming": "গ্লোবাল ওয়ার্মিং হ্রাসে উদ্যোগ নিন।",
    "sea_level_rising": "সমুদ্রপৃষ্ঠের উচ্চতা বৃদ্ধি রোধে ব্যবস্থা নিন।"
}
causes = list(cause2solution_bangla.keys())


# **Prediction function**

In [ ]:
import numpy as np

def predict_with_solutions(text, threshold=0.5):
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits.squeeze().cpu().numpy()

    # Multi-class softmax
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

    # Pick top cause
    top_idx = int(np.argmax(probs))
    top_cause = causes[top_idx] if top_idx < len(causes) else f"class_{top_idx}"
    top_score = float(probs[top_idx])

    return {
        "text": text,
        "predicted_causes": [(top_cause, top_score)],
        "solutions": {top_cause: cause2solution_bangla.get(top_cause, "🚧 No solution mapped.")}
    }


## **Example run**

In [ ]:
sample_text = "ঢাকায় যানবাহনের কারণে প্রচুর বায়ু দূষণ হচ্ছে।"
result = predict_with_solutions(sample_text)

print("📝 Input Text:", result["text"])
print("\n🔎 Predicted Causes & Scores:")
for cause, score in result["predicted_causes"]:
    print(f" - {cause}: {score:.2f}")

print("\n✅ Suggested Solutions:")
for cause, sol in result["solutions"].items():
    print(f" - {cause}: {sol}")


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


📝 Input Text: ঢাকায় যানবাহনের কারণে প্রচুর বায়ু দূষণ হচ্ছে।

🔎 Predicted Causes & Scores:
 - construction_noise: 0.05

✅ Suggested Solutions:
 - construction_noise: নির্মাণ সময় নির্দিষ্ট ঘণ্টায় কাজ করুন এবং শব্দ কমানোর যন্ত্র ব্যবহার করুন।
